### Preprocessing code adapted from  
- authors: Peitek Norman, Bergum Annabelle, Rekrut Maurice, Mucke Jonas, Nadig Matthias,Parnin Chris, Siegmund Janet, Apel Sven   
- title: "Correlates of Programmer Efficacy and Their Link to Experience: A Combined EEG and Eye-Tracking Study"  
- version: 1.0.0  
- date-released: 2022-10-01  
- url: "https://github.com/brains-on-code/NoviceVsExpert"  



In [16]:
import os
import re
import pandas as pd
from tqdm.notebook import tqdm
import mne
import re
import dask.dataframe as dd
import numpy as np
import sys

In [2]:
participants = []
eeg_raw_file_in = "C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_raw_data/"
for _dir, sub_dirs, _files in os.walk(eeg_raw_file_in):
    for dir in sub_dirs:
        numbers = re.findall(r'\d+', dir)
        participants.append(int(numbers[0]))
    break

In [8]:
ica_folder = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/ica"
raw_folder = "C:/Users/Mahima Acharya/Documents/BCI/Data/EEG_temp/raw"

# Move ICA eeg files back to the raw folder
for participant in tqdm(participants):
    participant_folder = "C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_raw_data/Participant" + str(participant).zfill(2)
    fdt_file_source = ica_folder + "/eeg_raw_" + str(participant).zfill(2) + ".fdt"
    set_file_source = ica_folder + "/eeg_raw_" + str(participant).zfill(2) + ".set"
    fdt_file_destination = participant_folder + "/eeg_raw_" + str(participant).zfill(2) + ".fdt"
    set_file_destination = participant_folder + "/eeg_raw_" + str(participant).zfill(2) + ".set"
    try:
        # check if the source file exists
        if os.path.exists(fdt_file_source):
            os.remove(fdt_file_destination)
    except:
        pass
    try:
        if os.path.exists(set_file_source):
            os.remove(set_file_destination)
    except:
        pass
    try:
        os.rename(fdt_file_source, fdt_file_destination)
        os.rename(set_file_source, set_file_destination)
    except:
        print("Participant " + str(participant) + " already has the files")

# delete every file in the raw_folder
for _dir, _sub_dirs, _files in os.walk(raw_folder):
    for file in _files:
        os.remove(raw_folder + "/" + file)

  0%|          | 0/39 [00:00<?, ?it/s]

Participant 1 already has the files
Participant 2 already has the files
Participant 3 already has the files
Participant 4 already has the files
Participant 5 already has the files
Participant 6 already has the files
Participant 7 already has the files
Participant 9 already has the files
Participant 10 already has the files
Participant 11 already has the files
Participant 12 already has the files
Participant 13 already has the files
Participant 14 already has the files
Participant 18 already has the files
Participant 22 already has the files
Participant 24 already has the files
Participant 25 already has the files
Participant 28 already has the files
Participant 35 already has the files
Participant 36 already has the files
Participant 37 already has the files
Participant 38 already has the files
Participant 41 already has the files
Participant 42 already has the files
Participant 46 already has the files
Participant 49 already has the files
Participant 50 already has the files
Participa

In [22]:


mne.set_log_level("WARNING")


def load_raw(participant_number, cores=12, digits=2, logging=True,
             raw_path="C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_raw_data/"):
    # setup output for logging
    output = sys.stdout
    if not logging:
        output = open(os.devnull, 'w')

    print("(01/10) Construct Paths", file=output, flush=True)
    # setup paths for loading
    participant_folder = raw_path + "Participant" + str(participant_number).zfill(digits) + "/"
    eeg_path = participant_folder + "eeg_raw.fif"
    eeg_set_path = participant_folder + "eeg_raw_" + str(participant_number).zfill(digits) + ".set"
    psychopy_csv_path = participant_folder + "experiment.csv"


    # Helper to read events from the info field directly; specific to some of our recordings
    def get_events_from_info(inst):
        eventsMNE = []
        eventsFromFIF = inst.info["events"]
        for event_idx in range(0, len(eventsFromFIF)):
            if eventsFromFIF[event_idx].get("list") is not None:
                content = eventsFromFIF[event_idx].get("list")
                content_list = content.tolist()
                content_new = [content_list[2], content_list[1], content_list[0]]
                eventsMNE.append(content_new)
            elif eventsFromFIF[event_idx].get("channels") is not None:
                raise
                # content = eventsFromFIF[i].get('channels')
            else:
                print("fiftools: Type of entry #" + str(event_idx + 1) + "unknown.")
        eventsMNE = np.array(eventsMNE)
        return eventsMNE

    print("(05/10) Read EEG Data", file=output, flush=True)
    # read the eeg data and scale it
    raw = mne.io.read_raw_fif(fname=eeg_path, preload=True)
    raw_set = mne.io.read_raw_eeglab(eeg_set_path, preload=True)

    print("(06/10) Construct Events from EEG Data", file=output, flush=True)
    # get the time of the events in seconds
    sampling_rate = raw.info["sfreq"]
    events = get_events_from_info(raw)
    event_ids = events[:, 2]
    indices_events = events[:, 0]
    t_events = event_ids / sampling_rate

    # save the event times in a dataframe for better handling
    columns = [
        "Snippet",
        "SnippetStart",
        "SnippetStop",
        "InputStart",
        "InputStop",
        "OutputStart",
        "OutputStop",
        "CrossStart",
        "CrossStop"
    ]
    df_time = pd.DataFrame([], columns=columns)
    for i in range(0, len(t_events)):
        if indices_events[i] > 100:
            continue
        #df_time = df_time.append(pd.DataFrame([[None, t_events[i + 1], t_events[i + 2], t_events[i + 2], t_events[i + 3], t_events[i + 3], None, None, None]], columns=columns, ))
        new_row = [None, t_events[i + 1], t_events[i + 2], t_events[i + 2], t_events[i + 3], t_events[i + 3], None, None, None]
        df_time = pd.concat([df_time, pd.DataFrame([new_row], columns=columns)], ignore_index=True)

    df_time = df_time.reset_index(drop=True)

    # extracts the file name from a path like "/test/path/file.txt" and returns "file"
    def to_file_name(path):
        file, _ext = os.path.splitext(path)
        return file.split("\\")[-1]

    # maps the answers from psychopy to better usable descriptors
    def map_to_answer(answer):
        answer = str(answer)  # Convert to string
        if "Right" in answer:
            return "Right"
        if "Wrong1" in answer:
            return "Wrong1"
        if "Wrong2" in answer:
            return "Wrong2"
        if "None" in answer:
            return "Wrong3"
        if "Skipped" in answer:
            return "Skipped"

    print("(07/10) Read PsychoPy Data", file=output, flush=True)
    # read the data from the psychopy csv file
    df_psydata = pd.read_csv(psychopy_csv_path)

    print("(08/10) Transform PsychoPy Data", file=output, flush=True)
    # create a dataframe which holds the times and answers given for each snippet
    df_psydata = df_psydata[
        ["ImagePath", "Image.started", "Image.stopped", "InputPath", "image.started", "image.stopped",
         "ImagePathInputs", "image_1.started", "image_1.stopped", "ChoosenAnwer", "image_7.started", ]
    ]
    df_psydata = df_psydata[df_psydata["ImagePath"].notna()]
    df_psydata.insert(0, "Snippet", df_psydata["ImagePath"].apply(to_file_name))
    df_psydata["ChoosenAnwer"] = df_psydata["ChoosenAnwer"].apply(map_to_answer)
    df_psydata = df_psydata.rename(columns={"ChoosenAnwer": "ChosenAnswer"})
    df_psydata = df_psydata.reset_index(drop=True)
    df_psydata = df_psydata.rename(columns={"Image.started": "SnippetStart", "Image.stopped": "SnippetStop"})
    df_psydata = df_psydata.rename(columns={"image.started": "InputStart", "image.stopped": "InputStop"})
    df_psydata = df_psydata.rename(columns={"image_1.started": "OutputStart", "image_1.stopped": "OutputStop"})
    df_psydata = df_psydata.rename(columns={"image_7.started": "CrossStart"})
    df_psydata["SnippetStop"] = df_psydata["InputStart"]
    df_psydata["InputStop"] = df_psydata["OutputStart"]
    df_psydata["OutputStop"] = df_psydata["CrossStart"]
    df_psydata = df_psydata.drop(["ImagePath", "InputPath", "ImagePathInputs", "CrossStart"], axis=1)

    print("(9/10) Normalize PsychoPy Time", file=output, flush=True)
    # normalize the time of all the snippets
    start_time = df_psydata["SnippetStart"][0]
    df_psydata["SnippetStart"] = df_psydata["SnippetStart"] - start_time
    df_psydata["SnippetStop"] = df_psydata["SnippetStop"] - start_time
    df_psydata["InputStart"] = df_psydata["InputStart"] - start_time
    df_psydata["InputStop"] = df_psydata["InputStop"] - start_time
    df_psydata["OutputStart"] = df_psydata["OutputStart"] - start_time
    df_psydata["OutputStop"] = df_psydata["OutputStop"] - start_time

    # set snippet name and the stop time of each snippet using deltas
    df_time["Snippet"] = df_psydata["Snippet"]
    df_time["OutputStop"] = df_time["OutputStart"] + df_psydata["OutputStop"] - df_psydata["OutputStart"]
    df_time["CrossStart"] = df_time["OutputStop"]
    df_time["CrossStop"] = df_time["CrossStart"] + 30

    # store all the data in a dictionary for better handling. split everything up by snippet
    result = {}

    # just the template to know the layout of the dictionary

    print("(10/10) Transform All Data to Dictionary", file=output, flush=True)
    # iterate for every snippet to set the data
    for index, row in df_psydata.iterrows():
        current = {"Code": { "EEG": None, "Time": {"Start": None, "Stop": None, }, },
                   "Input": {"EEG": None, "Time": {"Start": None, "Stop": None, }, },
                   "Output": { "EEG": None, "Time": {"Start": None, "Stop": None, }, },
                   "Cross": { "EEG": None, "Time": {"Start": None, "Stop": None, }, },
                   "Behavioral": None}

        # add data for code
        current["Code"]["EEG"] = raw_set.copy().crop(df_time["SnippetStart"][index], df_time["SnippetStop"][index])
        current["Code"]["Time"]["Start"] = df_psydata["SnippetStart"][index]
        current["Code"]["Time"]["Stop"] = df_psydata["SnippetStop"][index]

        # add data for input
        current["Input"]["EEG"] = raw_set.copy().crop(df_time["InputStart"][index], df_time["InputStop"][index])
        current["Input"]["Time"]["Start"] = df_psydata["InputStart"][index]
        current["Input"]["Time"]["Stop"] = df_psydata["InputStop"][index]

        # add data for input
        current["Output"]["EEG"] = raw_set.copy().crop(df_time["OutputStart"][index], df_time["OutputStop"][index])
        current["Output"]["Time"]["Start"] = df_psydata["OutputStart"][index]
        current["Output"]["Time"]["Stop"] = df_psydata["OutputStop"][index]

        current["Cross"]["EEG"] = raw_set.copy().crop(df_time["CrossStart"][index], df_time["CrossStop"][index])
        current["Cross"]["Time"]["Start"] = df_time["CrossStart"][index]
        current["Cross"]["Time"]["Stop"] = df_time["CrossStop"][index]

        current["Behavioral"] = df_psydata.iloc[index].to_frame().transpose()
        result[row["Snippet"]] = current

    return result

In [24]:
from contextlib import redirect_stderr, redirect_stdout

columns = ["Participant", "Algorithm", "Subpart", "Behavioral", "StartTime", "EndTime", "EEG", "CrossEEG"]
df_filtered = pd.DataFrame(columns=columns)

def rescale(data):
    # Scaling factor (to obtain values in [V], depends on device and settings etc.)
    scaling_factor = 1e-8
    return scaling_factor * data

# Iterate over all participants
for participant in tqdm(participants):
    if participant == 9:
        continue

    # Check if folder exists
    if not os.path.exists("C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant" + str(participant).zfill(2)):
        os.makedirs("C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant" + str(participant).zfill(2))

    # Load in Raw Data from Input folder
    # disables the stdout and stderr
    with open(os.devnull, 'w') as devnull:
        with redirect_stdout(devnull):
            with redirect_stderr(devnull):
                data = load_raw(participant, cores=24, logging=True)
    folder_prev = "C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/Participant" + str(participant).zfill(2) + "/"

    # save the raw data into splited data for task/input/output
    for algorithm in data.keys():
        # get the answer for the algorithm
        answer = data[algorithm]["Behavioral"]["ChosenAnswer"].array[0]

        # get the eeg data from cross fixation
        cross_eeg = data[algorithm]["Cross"]["EEG"]
        cross_eeg.apply_function(rescale, picks=['eeg'])
        cross_eeg.save(folder_prev + algorithm + "cross_eeg_raw.fif",overwrite=True)
        cross_eeg = folder_prev + algorithm + "cross_eeg_raw.fif"

        # get the start and end time, eyetracking and eeg data
        code_start = data[algorithm]["Behavioral"]["SnippetStart"].array[0]
        code_end = data[algorithm]["Behavioral"]["SnippetStop"].array[0]
        code_eeg = data[algorithm]["Code"]["EEG"]

        # rescale the eeg data
        code_eeg.apply_function(rescale, picks=['eeg'])

        #save code_eeg to file
        code_eeg.save(folder_prev + algorithm + "code_eeg_raw.fif", overwrite=True)
        code_eeg = folder_prev + algorithm + "code_eeg_raw.fif"


        # append the data to the dataframe
        df_filtered.loc[len(df_filtered)] = [
            participant, algorithm, "Code", answer,
            code_start, code_end,  code_eeg,
            cross_eeg]

        input_start = data[algorithm]["Behavioral"]["InputStart"].array[0]
        input_end = data[algorithm]["Behavioral"]["InputStop"].array[0]
        input_eeg = data[algorithm]["Input"]["EEG"]

        input_eeg.apply_function(rescale, picks=['eeg'])

        #save input_eeg to file
        input_eeg.save(folder_prev + algorithm + "input_eeg_raw.fif", overwrite=True)
        input_eeg = folder_prev + algorithm + "input_eeg_raw.fif"

        # append the data to the dataframe
        df_filtered.loc[len(df_filtered)] = [
            participant, algorithm, "Input", answer,
            input_start, input_end, input_eeg,
            cross_eeg]

        output_start = data[algorithm]["Behavioral"]["OutputStart"].array[0]
        output_end = data[algorithm]["Behavioral"]["OutputStop"].array[0]
        output_eeg = data[algorithm]["Output"]["EEG"]

        output_eeg.apply_function(rescale, picks=['eeg'])

        #save output_eeg to file
        output_eeg.save(folder_prev + algorithm + "output_eeg_raw.fif",overwrite=True)
        output_eeg = folder_prev + algorithm + "output_eeg_raw.fif"

        # append the data to the dataframe
        df_filtered.loc[len(df_filtered)] = [
            participant, algorithm, "Output", answer,
            output_start, output_end,  output_eeg,
            cross_eeg]

df_filtered.to_csv("C:/Users/Mahima Acharya/Documents/BCI/Data/eeg_filtered_data/filtered_data.csv", index=False)

  0%|          | 0/39 [00:00<?, ?it/s]